In [1]:
"""
Tiny Seq2Seq Encoder-Decoder Demo (LSTM based)
------------------------------------------------
Task: Learn to REVERSE a sequence of digits.
Example: input "1234" -> output "4321"
 
Why this task? It's small, synthetic (no dataset needed),
and clearly shows the encoder -> context vector -> decoder flow
you just learned after RNN/LSTM/GRU.
 
Architecture:
  Encoder: LSTM reads input sequence -> produces final hidden state (context)
  Decoder: LSTM takes context state -> generates output sequence step by step
"""
 
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [2]:
# -----------------------------
# 1. Create tiny synthetic data
# -----------------------------
NUM_SAMPLES = 2000
SEQ_LEN = 4                     # length of digit sequence, e.g. "1234"
VOCAB = list("0123456789")      # 10 possible tokens
CHAR2IDX = {c: i for i, c in enumerate(VOCAB)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
VOCAB_SIZE = len(VOCAB)
 
def generate_data(num_samples, seq_len):
    inputs, targets = [], []
    for _ in range(num_samples):
        seq = [np.random.randint(0, 10) for _ in range(seq_len)]
        inputs.append(seq)
        targets.append(seq[::-1])   # target = reversed sequence
    return np.array(inputs), np.array(targets)

In [3]:
 
X, y = generate_data(NUM_SAMPLES, SEQ_LEN)
 
# One-hot encode (since this is right after RNN/LSTM, keeping it simple —
# no embedding layer, just one-hot vectors per digit)
X_onehot = to_categorical(X, num_classes=VOCAB_SIZE)
y_onehot = to_categorical(y, num_classes=VOCAB_SIZE)
 

In [4]:
# Decoder input = target shifted right, with a "start token".
# We'll just use a zero vector as the start token for simplicity.
decoder_input = np.zeros_like(y_onehot)
decoder_input[:, 1:, :] = y_onehot[:, :-1, :]   # shift right by 1


In [5]:
# -----------------------------
# 2. Build Encoder-Decoder model
# -----------------------------
latent_dim = 64
 
# --- Encoder ---
encoder_inputs = Input(shape=(SEQ_LEN, VOCAB_SIZE), name="encoder_input")
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]   # this is the "context" passed to decoder
 

In [6]:
# --- Decoder ---
decoder_inputs = Input(shape=(SEQ_LEN, VOCAB_SIZE), name="decoder_input")
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(VOCAB_SIZE, activation="softmax", name="decoder_output")
decoder_outputs = decoder_dense(decoder_outputs)
 
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, 4, 10)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_input (InputLayer)    │ (None, 4, 10)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 64), (None, 64),  │          19,200 │ encoder_input[0][0]        │
│                               │ (None, 64)]               │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 4, 64), (None,    │          19,200 │ decoder_input[0][0],       │
│                               │ 64), (None, 64)]          │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_output (Dense)        │ (None, 4, 10)             │             650 │ decoder_lstm[0][0]         │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 39,050 (152.54 KB)

 Trainable params: 39,050 (152.54 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# -----------------------------
# 3. Train (small data -> trains fast)
# -----------------------------
model.fit(
    [X_onehot, decoder_input], y_onehot,
    batch_size=32,
    epochs=30,
    validation_split=0.1,
    verbose=2
)
 

Epoch 1/30
57/57 - 10s - 176ms/step - accuracy: 0.2857 - loss: 2.2363 - val_accuracy: 0.3487 - val_loss: 2.0903
Epoch 2/30
57/57 - 1s - 16ms/step - accuracy: 0.3750 - loss: 1.7192 - val_accuracy: 0.3900 - val_loss: 1.4340
Epoch 3/30
57/57 - 1s - 16ms/step - accuracy: 0.4324 - loss: 1.3039 - val_accuracy: 0.4663 - val_loss: 1.2049
Epoch 4/30
57/57 - 1s - 16ms/step - accuracy: 0.5496 - loss: 1.0775 - val_accuracy: 0.6363 - val_loss: 0.9712
Epoch 5/30
57/57 - 1s - 17ms/step - accuracy: 0.7260 - loss: 0.8227 - val_accuracy: 0.8150 - val_loss: 0.7083
Epoch 6/30
57/57 - 1s - 16ms/step - accuracy: 0.8717 - loss: 0.5680 - val_accuracy: 0.9013 - val_loss: 0.4994
Epoch 7/30
57/57 - 1s - 24ms/step - accuracy: 0.9442 - loss: 0.3869 - val_accuracy: 0.9475 - val_loss: 0.3497
Epoch 8/30
57/57 - 1s - 16ms/step - accuracy: 0.9799 - loss: 0.2636 - val_accuracy: 0.9787 - val_loss: 0.2371
Epoch 9/30
57/57 - 1s - 17ms/step - accuracy: 0.9919 - loss: 0.1912 - val_accuracy: 0.9925 - val_loss: 0.1764
Epoch 10

In [8]:
# -----------------------------
# 4. Inference: encoder model + decoder model (step-by-step generation)
# -----------------------------
encoder_model = Model(encoder_inputs, encoder_states)
 
decoder_input_inf = Input(shape=(None, VOCAB_SIZE), name="decoder_input_inf")  # 1 timestep at a time
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
 
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_input_inf, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
 
decoder_model = Model(
    [decoder_input_inf] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [9]:
def decode_sequence(input_seq_onehot):
    # Get context vector from encoder
    states_value = encoder_model.predict(input_seq_onehot, verbose=0)
 
    # Start with a zero vector (our "start token")
    target_seq = np.zeros((1, 1, VOCAB_SIZE))
 
    decoded = []
    for _ in range(SEQ_LEN):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_idx = np.argmax(output_tokens[0, 0, :])
        decoded.append(IDX2CHAR[sampled_idx])
 
        # Feed predicted token back as next input
        target_seq = np.zeros((1, 1, VOCAB_SIZE))
        target_seq[0, 0, sampled_idx] = 1.0
        states_value = [h, c]
 
    return "".join(decoded)
 

In [10]:
# -----------------------------
# 5. Test it on a few examples
# -----------------------------
print("\n--- Testing on new examples ---")
test_X, test_y = generate_data(5, SEQ_LEN)
test_X_onehot = to_categorical(test_X, num_classes=VOCAB_SIZE)
 
for i in range(5):
    input_str = "".join(str(d) for d in test_X[i])
    true_str = "".join(str(d) for d in test_y[i])
    pred_str = decode_sequence(test_X_onehot[i:i+1])
    print(f"Input: {input_str}  |  Expected: {true_str}  |  Predicted: {pred_str}")


--- Testing on new examples ---
Input: 8609  |  Expected: 9068  |  Predicted: 9068
Input: 6326  |  Expected: 6236  |  Predicted: 6236
Input: 2173  |  Expected: 3712  |  Predicted: 3712
Input: 1583  |  Expected: 3851  |  Predicted: 3851
Input: 5851  |  Expected: 1585  |  Predicted: 1585
